In [ ]:
import pandas as pd
from collections import Counter

In [ ]:
%run data_analysis.ipynb

## Conversion Labeling

We define two labeling strategies:

**Strategy A — Binary label (kept for reference):**  
`converted = 1` if the user ever visited the Checkout page.

**Strategy B — Funnel stage label (primary):**  
A 4-class label built from four independent conversion signals.  
Each signal contributes +1 to a composite score (0–4),  
which is then collapsed into four actionable funnel stages.

| Signal | What it detects | Strength |
|---|---|---|
| Checkout visit | Any intent to purchase | Weak — includes abandoners |
| Checkout + Coupon | Active inside purchase flow | Medium |
| Exit on Checkout | Session ended at purchase step | Medium |
| Certificate earned | Paid AND used the product | Strong |

| Stage | Score | Business label | Action |
|---|---|---|---|
| 0 | 0 | Browsing | Discovery content, awareness campaigns |
| 1 | 1 | Abandoned | Cart-recovery, retargeting |
| 2 | 2 | Interested | Discount nudge, limited-time offer |
| 3 | 3–4 | Converted | Upsell, cross-sell, retention |

In [ ]:
# ── Strategy A: Binary conversion label (baseline / reference) ────────────────

def label_conversions(data, target_column='user_journey', conversion_page='Checkout'):
    """
    Binary label: 1 if user visited the conversion page, 0 otherwise.
    Kept as baseline. Note: this includes checkout abandoners (noisy label).
    """
    data_copy = data.copy()
    data_copy['converted'] = (
        data_copy[target_column]
        .str.contains(conversion_page, case=False, na=False)
        .astype(int)
    )
    return data_copy


def get_conversion_summary(data, converted_column='converted'):
    total_users     = len(data)
    converted_users = data[converted_column].sum()
    not_converted   = total_users - converted_users
    conversion_rate = (converted_users / total_users) * 100 if total_users > 0 else 0
    return pd.DataFrame({
        'Metric': ['Total Users', 'Converted Users', 'Not Converted', 'Conversion Rate (%)'],
        'Value':  [total_users, converted_users, not_converted, round(conversion_rate, 2)]
    })

In [ ]:
# ── Strategy B: Multi-class funnel stage label (primary) ─────────────────────

# Funnel stage mapping
FUNNEL_STAGE_NAMES = {
    0: 'Browsing',
    1: 'Abandoned',
    2: 'Interested',
    3: 'Converted',
}


def label_funnel_stage(data, target_column='user_journey'):
    """
    Assign each user to a funnel stage using four independent conversion signals.

    Each signal contributes +1 to a composite score (0–4).
    The score is collapsed into four actionable business stages:

        Stage 0 — Browsing   (score 0): No purchase intent detected
        Stage 1 — Abandoned  (score 1): Visited Checkout, no corroboration
        Stage 2 — Interested (score 2): Two signals agree — moderate confidence
        Stage 3 — Converted  (score 3+): Three+ signals — high/definite confidence

    Signals:
        1. Visited Checkout at any point          (weak — includes abandoners)
        2. Visited Checkout AND Coupon            (medium — inside purchase flow)
        3. Journey exits on Checkout              (medium — session ended at purchase step)
        4. Earned a certificate                   (strong — paid AND used product)

    Scores 3 and 4 are collapsed into Stage 3 because score-4
    has only ~26 users — too sparse to be a separate class.

    Args:
        data          : DataFrame with grouped, cleaned user journeys
        target_column : column holding the journey string

    Returns:
        DataFrame with 'conversion_score' (0–4) and 'funnel_stage' (0–3) columns
    """
    df = data.copy()
    j  = df[target_column]

    # Signal 1: Visited Checkout at any point
    sig1 = j.str.contains('Checkout', case=False, na=False).astype(int)

    # Signal 2: Checkout + Coupon (deeper in purchase flow)
    sig2 = (
        j.str.contains('Checkout', case=False, na=False) &
        j.str.contains('Coupon',   case=False, na=False)
    ).astype(int)

    # Signal 3: Journey exits naturally on Checkout
    sig3 = j.str.endswith('Checkout').astype(int)

    # Signal 4: Earned a certificate — paid and engaged with the product
    sig4 = (
        j.str.contains('Career track certificate', case=False, na=False) |
        j.str.contains('Course certificate',       case=False, na=False)
    ).astype(int)

    df['conversion_score'] = sig1 + sig2 + sig3 + sig4

    # Collapse score into 4 stages (merge 3 and 4 — score-4 is only ~26 users)
    df['funnel_stage'] = df['conversion_score'].apply(
        lambda s: min(s, 3)
    )

    return df


def get_funnel_summary(data, stage_col='funnel_stage', score_col='conversion_score'):
    """
    Print a distribution table of funnel stages with business context.
    """
    total = len(data)
    rows  = []
    for stage in sorted(data[stage_col].unique()):
        n    = (data[stage_col] == stage).sum()
        name = FUNNEL_STAGE_NAMES.get(stage, f'Stage {stage}')
        rows.append({'Stage': stage, 'Label': name,
                     'Users': n, 'Pct (%)': round(n / total * 100, 1)})
    return pd.DataFrame(rows)

## Feature Engineering

### Leakage rules (updated for funnel stage label)

The funnel stage label is built from four signals.  
Any page that **directly encodes** one of those signals must be excluded from X:

| Page excluded | Why |
|---|---|
| `Checkout` | Signals 1, 2, 3 are all Checkout-derived |
| `Career track certificate` | Signal 4 — certificate = paid customer |
| `Course certificate` | Signal 4 — certificate = paid customer |

Note that `Coupon` is **kept** as a feature. A Coupon visit without Checkout  
does not reveal the label — its presence becomes a legitimate learned signal.

### Feature groups (65 features total)

| Group | Features | Description |
|---|---|---|
| Journey stats | 3 | Length, unique page count, session density |
| Page presence | 14 | Binary visit flag per safe page |
| Page counts | 14 | Visit frequency per safe page |
| Entry page | 14 | First page one-hot |
| Exit page | 14 | Last page one-hot (excl. all leaky pages) |
| Sequence/signals | 6 | Ordering and engagement patterns |
| Subscription | 2 | Annual / Monthly (Quarterly = baseline) |

In [ ]:
# ── Leakage constants ──────────────────────────────────────────────────────────

# Pages whose presence directly encodes the funnel stage label
# Checkout  → Signals 1, 2, 3
# Certificates → Signal 4
LEAKY_PAGES = {'Checkout', 'Career track certificate', 'Course certificate'}


def get_all_pages(data, target_column='user_journey'):
    """
    Scan the full dataset to collect every unique page name.
    Sorting ensures consistent column ordering across train/test splits.
    """
    pages = set()
    for journey in data[target_column].dropna():
        pages.update(journey.split('-'))
    return sorted(pages)


def _col(page):
    """Convert a page name to a safe column-name fragment."""
    return page.lower().replace(' ', '_').replace('-', '_')

In [ ]:
# ── Core Feature Extraction ───────────────────────────────────────────────────

def engineer_features(data, target_column='user_journey'):
    """
    Transform raw user journey strings into a structured numeric feature matrix.

    Excludes all pages in LEAKY_PAGES (Checkout + certificate pages) to
    prevent data leakage into the funnel stage label.

    Args:
        data          : DataFrame with user_journey (and optionally subscription_type)
        target_column : column holding the journey string

    Returns:
        pd.DataFrame  : One row per user, one column per feature. No label column.
    """
    df = data.copy()

    all_pages  = get_all_pages(df, target_column)
    safe_pages = [p for p in all_pages if p not in LEAKY_PAGES]

    feature_rows = []

    for _, row in df.iterrows():
        journey = row[target_column]
        pages   = journey.split('-') if pd.notna(journey) and journey else []
        counts  = Counter(pages)
        feat    = {}

        # ── 1. Journey-level statistics ───────────────────────────────────────
        feat['journey_length']    = len(pages)
        feat['unique_page_count'] = len(set(pages))
        # session_density: 1.0 = every visit was a new page (exploring)
        # near 0.0 = looping on same pages (stuck or highly focused)
        feat['session_density'] = (
            feat['unique_page_count'] / feat['journey_length']
            if feat['journey_length'] > 0 else 0
        )

        # ── 2. Page presence flags — binary (safe pages only) ─────────────────
        for page in safe_pages:
            feat[f'has_{_col(page)}'] = int(page in counts)

        # ── 3. Page visit counts (safe pages only) ────────────────────────────
        for page in safe_pages:
            feat[f'count_{_col(page)}'] = counts.get(page, 0)

        # ── 4. Entry page — one-hot ────────────────────────────────────────────
        entry_page = pages[0] if pages else None
        for page in safe_pages:
            feat[f'entry_{_col(page)}'] = int(entry_page == page)

        # ── 5. Exit page — one-hot (safe pages already exclude leaky pages) ────
        exit_page = pages[-1] if pages else None
        for page in safe_pages:
            feat[f'exit_{_col(page)}'] = int(exit_page == page)

        # ── 6. Sequence & behavioural signals ─────────────────────────────────

        # New user who registered this session vs returning user
        if 'Sign up' in counts and 'Log in' in counts:
            feat['signup_before_login'] = int(
                pages.index('Sign up') < pages.index('Log in')
            )
        else:
            feat['signup_before_login'] = 0

        # Career-content engagement — strong product intent signal
        feat['career_engaged'] = int(
            'Career tracks' in counts or 'Courses' in counts
        )

        # Pricing page — strongest pre-conversion indicator
        feat['visited_pricing']     = int('Pricing' in counts)
        feat['pricing_visit_count'] = counts.get('Pricing', 0)

        # Coupon page — signals deal-seeking behaviour
        # Note: Coupon is NOT in LEAKY_PAGES because Coupon alone (without Checkout)
        # does not reveal the label. It becomes a legitimate learned feature.
        feat['visited_coupon']     = int('Coupon' in counts)
        feat['coupon_visit_count'] = counts.get('Coupon', 0)

        # ── 7. Subscription type — one-hot ────────────────────────────────────
        # Quarterly is the baseline (omitted to avoid multicollinearity)
        if 'subscription_type' in row.index:
            sub = str(row['subscription_type']).lower()
            feat['sub_annual']  = int(sub == 'annual')
            feat['sub_monthly'] = int(sub == 'monthly')

        feature_rows.append(feat)

    return pd.DataFrame(feature_rows)

In [ ]:
# ── Feature Matrix Builders ───────────────────────────────────────────────────

def get_feature_matrix(labeled_data, target_column='user_journey', label_column='converted'):
    """
    Build X and y for binary classification (Strategy A — baseline).
    """
    X = engineer_features(labeled_data, target_column=target_column)
    y = labeled_data[label_column].reset_index(drop=True)
    return X, y, X.columns.tolist()


def get_funnel_matrix(labeled_data, target_column='user_journey', label_column='funnel_stage'):
    """
    Build X and y for multi-class funnel stage prediction (Strategy B — primary).

    y values:
        0 → Browsing   (no conversion signal)
        1 → Abandoned  (checkout visit only)
        2 → Interested (two signals — moderate confidence)
        3 → Converted  (three+ signals — high confidence)

    Returns:
        X             : pd.DataFrame — feature matrix (no label)
        y             : pd.Series   — funnel stage labels (0–3)
        feature_names : list[str]   — ordered feature column names
    """
    X = engineer_features(labeled_data, target_column=target_column)
    y = labeled_data[label_column].reset_index(drop=True)
    return X, y, X.columns.tolist()

In [ ]:
if __name__ == '__main__':

    # ── 1. Load & prepare data ────────────────────────────────────────────────
    data = pd.read_csv('user_journey_raw.csv')

    subscription_map = (
        data.groupby('user_id')['subscription_type']
        .first().reset_index()
    )

    grouped_data = group_by(data)
    grouped_data = grouped_data.merge(subscription_map, on='user_id', how='left')
    cleaned_data = remove_page_duplicates(grouped_data, 'user_journey')

    # ── 2. Apply BOTH labeling strategies ─────────────────────────────────────
    # Strategy A: binary
    labeled_binary = label_conversions(cleaned_data, conversion_page='Checkout')

    # Strategy B: funnel stages (primary)
    labeled_funnel = label_funnel_stage(cleaned_data)

    print('=== Strategy A — Binary label (baseline) ===')
    print(get_conversion_summary(labeled_binary).to_string(index=False))

    print('\n=== Strategy B — Funnel stage label (primary) ===')
    print(get_funnel_summary(labeled_funnel).to_string(index=False))

    print('\n  Score distribution (before collapsing):')
    score_counts = labeled_funnel['conversion_score'].value_counts().sort_index()
    for score, count in score_counts.items():
        bar = '█' * int(count / len(labeled_funnel) * 30)
        print(f'    Score {score}: {count:>4} users ({count/len(labeled_funnel)*100:4.1f}%)  {bar}')

    # ── 3. Engineer features ──────────────────────────────────────────────────
    X, y, feature_names = get_funnel_matrix(labeled_funnel)

    print(f'\n=== Feature Matrix ===')
    print(f'Shape : {X.shape[0]} users  x  {X.shape[1]} features')

    groups = {
        'Journey stats   ': [f for f in feature_names
                              if f in ['journey_length', 'unique_page_count', 'session_density']],
        'Page presence   ': [f for f in feature_names if f.startswith('has_')],
        'Page counts     ': [f for f in feature_names if f.startswith('count_')],
        'Entry page      ': [f for f in feature_names if f.startswith('entry_')],
        'Exit page       ': [f for f in feature_names if f.startswith('exit_')],
        'Sequence/signals': [f for f in feature_names
                              if f in ['signup_before_login', 'career_engaged',
                                       'visited_pricing', 'pricing_visit_count',
                                       'visited_coupon',   'coupon_visit_count',
                                       'sub_annual',       'sub_monthly']],
    }
    print()
    for g, cols in groups.items():
        print(f'  {g}: {len(cols)} features')

    print(f'\n=== Data quality checks ===')
    print(f'  Nulls in X                   : {X.isnull().sum().sum()}')
    print(f'  Nulls in y                   : {y.isnull().sum()}')
    print(f'  Dtypes                       : {dict(X.dtypes.value_counts())}')
    print(f'  has_checkout in X            : {"has_checkout" in feature_names}  (must be False)')
    print(f'  has_career_track_cert in X   : {"has_career_track_certificate" in feature_names}  (must be False)')
    print(f'  has_course_certificate in X  : {"has_course_certificate" in feature_names}  (must be False)')

    print(f'\n=== Sample — first 5 users with funnel stage ===')
    sample_cols = [
        'journey_length', 'unique_page_count', 'session_density',
        'has_pricing', 'has_coupon', 'has_sign_up', 'has_log_in',
        'career_engaged', 'visited_pricing', 'visited_coupon', 'sub_annual'
    ]
    sample = X[sample_cols].head(5).copy()
    sample['score']        = labeled_funnel['conversion_score'].values[:5]
    sample['funnel_stage'] = y.head(5).values
    sample['stage_label']  = sample['funnel_stage'].map(FUNNEL_STAGE_NAMES)
    print(sample.to_string())

    print(f'\n=== All {len(feature_names)} features ===')
    for i, name in enumerate(feature_names):
        print(f'  [{i:02d}] {name}')